In [13]:
%pip install pandas numpy lightgbm scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\tech\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd
import numpy as np
import joblib
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [ ]:
#train = pd.read_parquet('../artifacts/train_fe.parquet')
#val = pd.read_parquet('../artifacts/val_fe.parquet')
#test = pd.read_parquet('../artifacts/test_fe.parquet')

In [ ]:
train = pd.read_parquet('../data/processed/train_fe.parquet')
val = pd.read_parquet('../data/processed/val_fe.parquet')
test = pd.read_parquet('../data/processed/test_fe.parquet')

In [ ]:
#features = joblib.load('../artifacts/feature_cols.joblib')
#target = 'is_late'

In [ ]:
features = joblib.load('../models/feature_cols.joblib')
target = 'is_late'

In [17]:
X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]
X_test, y_test = test[features], test[target]

In [19]:
model = LGBMClassifier(
    n_estimators=150,
    learning_rate=0.03,
    max_depth=6,
    scale_pos_weight=11,
    random_state=42
)

In [6]:
model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    scale_pos_weight=11,
    random_state=42
)

In [22]:
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 5479, number of negative: 62054
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001317 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1106
[LightGBM] [Info] Number of data points in the train set: 67533, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.081131 -> initscore=-2.427082
[LightGBM] [Info] Start training from score -2.427082
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


,max_depth,6
,learning_rate,0.03
,n_estimators,150
,random_state,42
,scale_pos_weight,11
,boosting_type,'gbdt'
,num_leaves,31
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [23]:
y_val_pred = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)[:, 1]

In [24]:
val_roc = roc_auc_score(y_val, y_val_proba)

In [25]:
print(classification_report(y_val, y_val_pred))
print(f"Validation ROC-AUC Score: {val_roc:.4f}")

              precision    recall  f1-score   support

           0       0.92      1.00      0.96     13297
           1       0.00      0.00      0.00      1174

    accuracy                           0.92     14471
   macro avg       0.46      0.50      0.48     14471
weighted avg       0.84      0.92      0.88     14471

Validation ROC-AUC Score: 0.5402


C:\Users\tech\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\tech\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\tech\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [26]:
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]

In [27]:
test_roc = roc_auc_score(y_test, y_test_proba)

In [28]:
print(classification_report(y_test, y_test_pred))
print(f"Final Test ROC-AUC Score: {test_roc:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.92      1.00      0.96     13298
           1       0.00      0.00      0.00      1174

    accuracy                           0.92     14472
   macro avg       0.46      0.50      0.48     14472
weighted avg       0.84      0.92      0.88     14472

Final Test ROC-AUC Score: 0.5302

Confusion Matrix:
[[13298     0]
 [ 1174     0]]


C:\Users\tech\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\tech\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\tech\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [29]:
train = pd.read_parquet('../artifacts/train.parquet')
val = pd.read_parquet('../artifacts/val.parquet')
test = pd.read_parquet('../artifacts/test.parquet')

In [31]:
def prepare_data(df):
    data = df.copy()
    data['order_purchase_timestamp'] = pd.to_datetime(data['order_purchase_timestamp'])
    data['order_estimated_delivery_date'] = pd.to_datetime(data['order_estimated_delivery_date'])

    data['purchase_year'] = data['order_purchase_timestamp'].dt.year
    data['purchase_month'] = data['order_purchase_timestamp'].dt.month
    data['purchase_dayofweek'] = data['order_purchase_timestamp'].dt.dayofweek
    data['purchase_hour'] = data['order_purchase_timestamp'].dt.hour

    data['estimated_delivery_days'] = (
        data['order_estimated_delivery_date'] - data['order_purchase_timestamp']
    ).dt.total_seconds() / (24 * 3600)

    feats = ['purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour',
             'estimated_delivery_days', 'total_price', 'total_freight',
             'total_items', 'total_payment', 'payment_installments']

    return data[feats].fillna(0), data['is_late']

In [32]:
X_train, y_train = prepare_data(train)
X_val, y_val = prepare_data(val)
X_test, y_test = prepare_data(test)

In [33]:
model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    scale_pos_weight=11,
    random_state=42
)

In [34]:
model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 5479, number of negative: 62054
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001378 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1104
[LightGBM] [Info] Number of data points in the train set: 67533, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.081131 -> initscore=-2.427082
[LightGBM] [Info] Start training from score -2.427082


,learning_rate,0.05
,random_state,42
,scale_pos_weight,11
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [35]:
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]

In [36]:
print(classification_report(y_test, y_test_pred))
print(f"Final Test ROC-AUC Score: {roc_auc_score(y_test, y_test_proba):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.96      0.74      0.84     13298
           1       0.18      0.63      0.28      1174

    accuracy                           0.73     14472
   macro avg       0.57      0.69      0.56     14472
weighted avg       0.89      0.73      0.79     14472

Final Test ROC-AUC Score: 0.7455

Confusion Matrix:
[[9882 3416]
 [ 435  739]]


In [ ]:
import joblib
#model_path = '../artifacts/model.joblib'
#joblib.dump(model, model_path)

['../artifacts/model.joblib']

In [ ]:
os.makedirs('../models', exist_ok=True)
model_path = '../models/model.joblib'
joblib.dump(model, model_path)

In [39]:

summary = f"""=== MLOps Task 2: Model Results Summary ===
- Final Test ROC-AUC: {roc_auc_score(y_test, y_test_proba):.4f}
- Class 1 (Late Delivery) Recall: 0.63
- Class 1 Precision: 0.18
- Accuracy: 0.73
- Artifact Saved: {model_path}
"""

In [41]:
with open('../artifacts/model_results.txt', 'w', encoding='utf-8') as f:
    f.write(summary)